# 03 — rapids-singlecell GPU Harmonized: Mouse Brain 1M

**Run on Libra H100 GPU.**

Subsets to 1,000,000 cells to stay under CuPy's int32 sparse index ceiling (~2.1B nnz).
This matches the official rsc paper (Dicks et al. 2026) which benchmarked on 1M cells
from the same dataset.

Scanpy CPU ran on the full 1,153,539 cells. Report the difference in cell count in the paper.

In [1]:
import json, time, os, inspect
import numpy as np
import pandas as pd
import cupy as cp
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator

rmm.reinitialize(pool_allocator=True, managed_memory=True)
cp.cuda.set_allocator(rmm_cupy_allocator)

import rapids_singlecell as rsc
import scanpy as sc

sc.settings.verbosity = 0
print(f"rsc: {rsc.__version__}")
print(f"Scanpy: {sc.__version__}")
os.makedirs("write", exist_ok=True)

with open("benchmark_config.json") as f:
    cfg = json.load(f)
gcfg = cfg["global"]
dcfg = cfg["datasets"]["mouse_brain_1m"]
prefix = dcfg["pipeline_prefix"]
timings = {}

rsc: 0.14.1
Scanpy: 1.12


/tmp/ipykernel_1185707/3930674277.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(f"Scanpy: {sc.__version__}")


## Load + Subset to 1M + GPU transfer

In [2]:
t0 = time.time()
adata = sc.read_h5ad(dcfg["canonical_h5ad"])
timings["load"] = time.time() - t0
print(f"Loaded in {timings['load']:.1f}s: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
assert "counts" in adata.layers

# Subset to 1M cells — matches rsc paper (Dicks et al. 2026)
N_CELLS = 1_000_000
full_n_obs = adata.n_obs
if adata.n_obs > N_CELLS:
    print(f"Subsetting to {N_CELLS:,} cells (from {adata.n_obs:,})")
    adata = adata[:N_CELLS, :].copy()
    print(f"After subset: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
    print(f"nnz after subset: {adata.X.nnz:,} (INT32_MAX: {2**31-1:,})")

t0 = time.time()
rsc.get.anndata_to_GPU(adata)
timings["to_gpu"] = time.time() - t0
print(f"Moved to GPU in {timings['to_gpu']:.1f}s")
print(f"indptr dtype: {adata.X.indptr.dtype}, indices dtype: {adata.X.indices.dtype}")

Loaded in 90.1s: 1,153,539 cells x 22,788 genes
Subsetting to 1,000,000 cells (from 1,153,539)
After subset: 1,000,000 cells x 22,788 genes
nnz after subset: 2,115,352,065 (INT32_MAX: 2,147,483,647)
Moved to GPU in 7.1s
indptr dtype: int32, indices dtype: int32


## Normalize + log1p

In [3]:
t0 = time.time()
rsc.pp.normalize_total(adata, target_sum=gcfg["target_sum"])
rsc.pp.log1p(adata)
cp.cuda.Device(0).synchronize()
timings["normalize_log1p"] = time.time() - t0
print(f"Normalize + log1p: {timings['normalize_log1p']:.1f}s")

Normalize + log1p: 4.7s


## HVG

In [4]:
import cupyx.scipy.sparse as cusp
from scipy import sparse as sp

# Ensure counts layer is on GPU
if sp.issparse(adata.layers["counts"]):
    print("Converting counts layer to GPU...")
    adata.layers["counts"] = cusp.csr_matrix(adata.layers["counts"])

t0 = time.time()
rsc.pp.highly_variable_genes(
    adata, layer="counts", n_top_genes=dcfg["n_top_genes"],
    flavor="seurat_v3",
)
timings["hvg"] = time.time() - t0
print(f"HVG ({adata.var['highly_variable'].sum()} genes): {timings['hvg']:.1f}s")

Converting counts layer to GPU...
HVG (5000 genes): 8.1s


## PCA

In [5]:
t0 = time.time()
rsc.pp.pca(adata, n_comps=gcfg["pca_n_comps"], mask_var="highly_variable")
timings["pca"] = time.time() - t0
print(f"PCA: {timings['pca']:.1f}s")

PCA: 3.7s


## Neighbors

In [6]:
t0 = time.time()
rsc.pp.neighbors(
    adata, n_neighbors=dcfg["n_neighbors"], n_pcs=dcfg["neighbors_n_pcs"],
    use_rep="X_pca", algorithm="brute",
    metric=gcfg["neighbor_metric"], random_state=gcfg["random_state"],
)
timings["neighbors"] = time.time() - t0
print(f"Neighbors: {timings['neighbors']:.1f}s")

Neighbors: 13.0s


## Leiden

In [7]:
t0 = time.time()
leiden_kwargs = {
    "resolution": dcfg["leiden_resolution"],
    "random_state": gcfg["random_state"],
    "key_added": "leiden",
}
if gcfg.get("rsc_leiden_n_iterations") is not None:
    leiden_kwargs["n_iterations"] = gcfg["rsc_leiden_n_iterations"]
rsc.tl.leiden(adata, **leiden_kwargs)
timings["leiden"] = time.time() - t0
n_clusters = adata.obs["leiden"].nunique()
print(f"Leiden ({n_clusters} clusters): {timings['leiden']:.1f}s")

Leiden (30 clusters): 3.2s


## Checkpoint: save clusters

In [8]:
rsc.get.anndata_to_CPU(adata)
out = f"write/{prefix}_rsc_gpu_harmonized"
pd.DataFrame({
    "barcode": adata.obs_names.astype(str),
    "leiden": adata.obs["leiden"].astype(str).values,
}).to_csv(f"{out}_clusters.csv", index=False)
rsc.get.anndata_to_GPU(adata)
print(f"Checkpoint: clusters saved ({n_clusters} clusters)")

Checkpoint: clusters saved (30 clusters)


## UMAP

In [9]:
t0 = time.time()
rsc.tl.umap(
    adata, min_dist=gcfg["umap_min_dist"], spread=gcfg["umap_spread"],
    init_pos="spectral", random_state=gcfg["random_state"],
)
timings["umap"] = time.time() - t0
print(f"UMAP: {timings['umap']:.1f}s")

UMAP: 3.8s


## Differential Expression

In [10]:
t0 = time.time()
rank_sig = inspect.signature(rsc.tl.rank_genes_groups)
rank_kwargs = {
    "groupby": "leiden",
    "method": gcfg["de_method"],
    "corr_method": gcfg["de_corr_method"],
    "use_raw": False,
    "pts": True,
}
for key, value in {"pre_load": True, "tie_correct": False, "use_continuity": False}.items():
    if key in rank_sig.parameters:
        rank_kwargs[key] = value
rsc.tl.rank_genes_groups(adata, **rank_kwargs)
timings["de"] = time.time() - t0
print(f"DE: {timings['de']:.1f}s")

DE: 38.1s


## Move back to CPU + Save all outputs

In [11]:
t0 = time.time()
rsc.get.anndata_to_CPU(adata)
timings["to_cpu"] = time.time() - t0

out = f"write/{prefix}_rsc_gpu_harmonized"

# Clusters
pd.DataFrame({
    "barcode": adata.obs_names.astype(str),
    "leiden": adata.obs["leiden"].astype(str).values,
}).to_csv(f"{out}_clusters.csv", index=False)

# Markers
markers = sc.get.rank_genes_groups_df(adata, group=None)
markers.to_csv(f"{out}_markers.csv", index=False)
markers_filt = markers[(markers["pvals_adj"] < 0.05) & (markers["logfoldchanges"] > 0.1)]
markers_filt.to_csv(f"{out}_markers_filtered.csv", index=False)

# UMAP
pd.DataFrame(
    adata.obsm["X_umap"], index=adata.obs_names.astype(str),
    columns=["UMAP_1", "UMAP_2"],
).reset_index(names="barcode").to_csv(f"{out}_umap.csv", index=False)

# HVG
adata.var.loc[adata.var["highly_variable"].astype(bool), []].reset_index(
    names="gene"
).to_csv(f"{out}_hvg.csv", index=False)

# h5ad
t0s = time.time()
adata.write_h5ad(f"{out}.h5ad", compression="gzip")
timings["save_h5ad"] = time.time() - t0s

# Spec
spec = {
    "pipeline": "rsc_gpu_harmonized",
    "dataset": dcfg["name"],
    "rapids_singlecell_version": rsc.__version__,
    "cell_subset": {
        "n_cells_used": N_CELLS,
        "n_cells_total": full_n_obs,
        "reason": "CuPy int32 sparse index ceiling at ~2.1B nnz; matches rsc paper (Dicks et al. 2026)",
    },
    "timings_seconds": timings,
    "results": {
        "n_cells": int(adata.n_obs),
        "n_genes": int(adata.n_vars),
        "n_hvg": int(adata.var["highly_variable"].sum()),
        "n_clusters": n_clusters,
        "n_marker_rows": len(markers),
        "n_marker_rows_filtered": len(markers_filt),
    },
}
with open(f"{out}_spec.json", "w") as f:
    json.dump(spec, f, indent=2)

print("\n=== Timings ===")
total = 0
for step, t in timings.items():
    print(f"  {step:20s}: {t:8.1f}s")
    total += t
print(f"  {'TOTAL':20s}: {total:8.1f}s ({total/60:.1f} min)")
print(f"\nCells: {adata.n_obs:,} (of {full_n_obs:,} total)")
print(f"Clusters: {n_clusters}")
print(f"Filtered markers: {len(markers_filt)}")


=== Timings ===
  load                :     90.1s
  to_gpu              :      7.1s
  normalize_log1p     :      4.7s
  hvg                 :      8.1s
  pca                 :      3.7s
  neighbors           :     13.0s
  leiden              :      3.2s
  umap                :      3.8s
  de                  :     38.1s
  to_cpu              :      3.4s
  save_h5ad           :    316.5s
  TOTAL               :    491.8s (8.2 min)

Cells: 1,000,000 (of 1,153,539 total)
Clusters: 30
Filtered markers: 126750
